In [ ]:
from jax import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import numpyro
from numpyro import distributions as dist
from numpyro import infer
import arviz as az
from pathlib import Path
import plotly.graph_objects as go
import seaborn as sns
import pickle

from estival.sampling import tools as esamp

from emu_renewal.inputs import DATA_PATH, MOB_MAP, get_multicountry_df_from_who_data, get_hosp_series_from_owid_data, get_multicountry_df_from_who_data
from emu_renewal.inputs import get_seroprev, filter_seroprev
from emu_renewal.inputs import get_multivars_country_data, get_row_proportions, VAR_MAP
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import get_spaghetti, get_spagh_df_from_dict, get_combined_df, get_df_from_3darray, get_output_dir
from emu_renewal.plotting import plot_proc_comparison, plot_post_prior_comparison
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
analysis = "no_mob"
country = "France"

In [ ]:
# Analysis type
mobility = analysis == "mob"

# WHO indicators
cases_data = get_multicountry_df_from_who_data("New_cases", [country])
deaths_data = get_multicountry_df_from_who_data("New_deaths", [country])

# Hospitalisation data
hosp_data = get_hosp_series_from_owid_data("Daily hospital occupancy", country)

# Variant data
var_data = get_row_proportions(get_multivars_country_data(VAR_MAP, country))
var_data["eu"] = var_data["eu1"] + var_data["eu2"]
var_data = var_data[["eu", "alpha"]]

# Mobility
mob = pd.read_csv(DATA_PATH / f"mobility/{MOB_MAP[country]}_mob_data.csv", index_col=0)
mob.index = pd.to_datetime(mob.index)
non_resi_mob = mob.loc[:, mob.columns!="residential"]
model_mob = non_resi_mob.mean(axis=1).rolling(7).mean().dropna() if mobility else None

In [ ]:
# Specify fixed parameters and get calibration data
proc_update_freq = 14
init_time = 50
pop = 26e6
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
data_start = analysis_start + timedelta(14)
init_start = analysis_start - timedelta(init_time)
init_end = analysis_start - timedelta(1)
select_data = cases_data.loc[data_start: analysis_end, country]
select_deaths = deaths_data.loc[data_start: analysis_end, country]
select_vars = var_data.loc[data_start: analysis_end]
hosp_data = hosp_data[data_start: analysis_end: 7]
init_data = cases_data[country].resample("D").asfreq().interpolate().loc[init_start: init_end] / 7.0

In [ ]:
seroprev = filter_seroprev(get_seroprev(), country, analysis_start, analysis_end)
seroprev.index -= timedelta(days=14)

In [ ]:
# Define model and fitter
alpha_seed_times = [datetime(2020, 9, 15), datetime(2020, 9, 20)]
seed_times = [alpha_seed_times]
proc_fitter = CosineMultiCurve()
strains = ["eu", "alpha"]
model_args = [
    pop, 
    analysis_start, 
    analysis_end, 
    proc_update_freq, 
    proc_fitter, 
    GammaDens(), 
    init_time, 
    init_data, 
    GammaDens(), 
    GammaDens(), 
    strains, 
    "eu", 
    seed_times, 
    20.0, 
]
model = MultiStrainModel(*model_args + [model_mob])

In [ ]:
# Define parameter ranges
priors = {
    "gen_mean": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "gen_sd": dist.TruncatedNormal(2.5, 0.5, low=1.0),
    "cdr": dist.Beta(10, 25),
    "ifr": dist.Beta(3, 200),
    "rt_init": dist.Normal(0.0, 0.25),
    "report_mean": dist.TruncatedNormal(8.0, 0.5, low=1.0),
    "report_sd": dist.TruncatedNormal(3.0, 0.5, low=1.0),
    "death_mean": dist.TruncatedNormal(25.0, 5.0, low=1.0),
    "death_sd": dist.TruncatedNormal(10.0, 2.5, low=1.0),
    "admit_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "admit_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "stay_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "stay_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "har": dist.Beta(5, 200),
    "shared_dispersion": dist.HalfNormal(0.5),
    "cross_immunity": dist.Beta(25, 20),
}

In [ ]:
# Define calibration and calib data
calib_data = {
    "weekly_cases": StandardDispTarget(select_data, weight=20.0),
    # "weekly_deaths": StandardDispTarget(select_deaths, weight=20.0),
    "prop_eu": StandardDispTarget(var_data.loc[(datetime(2020, 9, 1) < var_data.index) & (var_data.index < datetime(2021, 7, 1)), "eu"], weight=20.0),
    "seropos": StandardDispTarget(seroprev, weight=20.0),
    # "occupancy": StandardDispTarget(hosp_data, weight=20.0),
}
calib = StandardCalib(model, priors, calib_data)

In [ ]:
# Run calibration
analysis_time = datetime.now().strftime("%d%m%Y_%H%M")
kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.5))
mcmc = infer.MCMC(kernel, num_chains=2, num_samples=200, num_warmup=200)
mcmc.run(random.PRNGKey(1))

In [ ]:
out_dir = get_output_dir(country, analysis, analysis_time)

In [ ]:
# Get and store outputs
idata = az.from_dict(mcmc.get_samples(True))
idata.to_netcdf(out_dir / "idata.nc")
idata_sampled = az.extract(idata, num_samples=20)
sample_params = esamp.xarray_to_sampleiterator(idata_sampled)
spaghetti = get_spagh_df_from_dict(get_spaghetti(calib, sample_params))
spaghetti.to_hdf(out_dir / "spaghetti.h5", key="s")
updates = pd.DataFrame(sample_params.components["proc"], columns=model.epoch.index_to_dti(model.x_proc_vals)).T
updates.to_hdf(out_dir / "updates.h5", key="u")
pickle.dump(priors, open(get_output_dir(country, analysis, analysis_time) / "priors.pkl", "wb"))
for t, target in calib.targets.items():
    target.data.to_hdf(out_dir / f"target_{t}.h5", key=t)

In [ ]:
from jax import numpy as jnp

In [ ]:
window_len = 50
t = 10

In [ ]:
seed_start = t - model.window_len
extra_zeroes = jnp.zeros([model.n_strains, max([-seed_start, 0])])
seed_vals = model.seeding[:, max([seed_start, 0]) + model.init_length: t + model.init_length]
seeding = jnp.concat([extra_zeroes, seed_vals], axis=1)
seed_inc = jnp.fliplr(seeding)